# 3.2 线性注意力与亚二次复杂度架构

## 什么是线性注意力？

标准注意力的$O(n^2)$复杂度限制了长序列推理。线性注意力通过kernel化或分解将复杂度降至$O(n)$，状态空间模型（SSM）通过隐状态递推实现$O(n)$推理。

## 为什么需要线性注意力？

1. **长序列瓶颈**：标准注意力的$O(n^2)$复杂度使得序列长度超过一定阈值后，计算量和内存急剧增长。
2. **端侧内存限制**：端侧设备无法存储$n \times n$的注意力矩阵，线性复杂度是必需的。
3. **实时性要求**：流式处理（如语音识别）需要恒定时间的每步推理。

## 线性注意力的核心思想

标准注意力：$\text{Attn}(Q,K,V) = \text{softmax}(QK^T/\sqrt{d})V$，先算$QK^T$（$O(n^2)$）

线性注意力：$\text{LinAttn}(Q,K,V) = \frac{\phi(Q)(\phi(K)^T V)}{\phi(Q)(\phi(K)^T \mathbf{1})}$，先算$\phi(K)^T V$（$O(nd^2)$）

其中$\phi$是特征映射函数，将softmax的指数函数替换为可分解的核函数。当$d \ll n$时，复杂度从$O(n^2)$降至$O(nd^2)$。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time

torch.manual_seed(42)
np.random.seed(42)
print(f"PyTorch version: {torch.__version__}")

---
### 线性注意力（Linear Attention）

#### 什么是线性注意力？

将softmax注意力中的指数核替换为可分解的特征映射$\phi$，改变计算顺序使复杂度从$O(n^2)$降至$O(nd^2)$。

#### 为什么线性注意力有效？

1. **计算顺序变换**：标准注意力先算$QK^T$（$O(n^2)$），线性注意力先算$\phi(K)^T V$（$O(nd^2)$）
2. **递推计算**：$\phi(K)^T V$可以用递推更新，每步$O(d^2)$
3. **长序列友好**：当$n \gg d$时，$O(nd^2) \ll O(n^2)$

#### 线性注意力的数学原理

标准注意力：$\text{Attn}(Q,K,V) = \text{softmax}(QK^T/\sqrt{d})V$

线性注意力：
$$\text{LinAttn}(Q,K,V) = \frac{\phi(Q)(\phi(K)^T V)}{\phi(Q)(\phi(K)^T \mathbf{1})}$$

其中$\phi$为特征映射函数，常见选择：
- **ELU+1**：$\phi(x) = \text{ELU}(x) + 1$
- **ReLU**：$\phi(x) = \text{ReLU}(x)$
- **Performer**：$\phi(x) = \frac{1}{\sqrt{m}}e^{iWx}$（随机正交投影）

关键：$\phi(K)^T V \in \mathbb{R}^{d \times d}$与序列长度$n$无关，可递推更新

In [ ]:
class LinearAttention(nn.Module):
    def __init__(self, dim, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.q_proj = nn.Linear(dim, dim, bias=False)
        self.k_proj = nn.Linear(dim, dim, bias=False)
        self.v_proj = nn.Linear(dim, dim, bias=False)
        self.out_proj = nn.Linear(dim, dim, bias=False)
        self.eps = 1e-6

    def feature_map(self, x):
        return F.elu(x) + 1

    def forward(self, x):
        B, N, C = x.shape
        q = self.q_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)

        q_feat = self.feature_map(q)
        k_feat = self.feature_map(k)

        kv = torch.einsum('bhnd,bhne->bhde', k_feat, v)
        qkv = torch.einsum('bhnd,bhde->bhne', q_feat, kv)

        k_sum = k_feat.sum(dim=2)
        normalizer = torch.einsum('bhnd,bhd->bhn', q_feat, k_sum).unsqueeze(-1).clamp(min=self.eps)

        out = qkv / normalizer
        out = out.transpose(1, 2).reshape(B, N, C)
        return self.out_proj(out)

class LinearAttentionRecurrent(nn.Module):
    def __init__(self, dim, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.q_proj = nn.Linear(dim, dim, bias=False)
        self.k_proj = nn.Linear(dim, dim, bias=False)
        self.v_proj = nn.Linear(dim, dim, bias=False)
        self.out_proj = nn.Linear(dim, dim, bias=False)
        self.eps = 1e-6

    def feature_map(self, x):
        return F.elu(x) + 1

    def forward(self, x):
        B, N, C = x.shape
        q = self.q_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)

        q_feat = self.feature_map(q)
        k_feat = self.feature_map(k)

        S = torch.zeros(B, self.n_heads, self.head_dim, self.head_dim, device=x.device)
        z = torch.zeros(B, self.n_heads, self.head_dim, device=x.device)
        outputs = []

        for t in range(N):
            k_t = k_feat[:, :, t, :]
            v_t = v[:, :, t, :]
            q_t = q_feat[:, :, t, :]
            S = S + torch.einsum('bhd,bhe->bhde', k_t, v_t)
            z = z + k_t
            num = torch.einsum('bhd,bhde->bhe', q_t, S)
            den = (q_t * z).sum(dim=-1, keepdim=True).clamp(min=self.eps)
            o_t = num / den
            outputs.append(o_t)

        out = torch.stack(outputs, dim=2).transpose(1, 2).reshape(B, N, C)
        return self.out_proj(out)

class StandardAttention(nn.Module):
    def __init__(self, dim, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.q_proj = nn.Linear(dim, dim, bias=False)
        self.k_proj = nn.Linear(dim, dim, bias=False)
        self.v_proj = nn.Linear(dim, dim, bias=False)
        self.out_proj = nn.Linear(dim, dim, bias=False)

    def forward(self, x):
        B, N, C = x.shape
        q = self.q_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
        attn = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)
        attn = attn.softmax(dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.out_proj(out)

dim, n_heads = 128, 4
linear_attn = LinearAttention(dim, n_heads)
linear_attn_rec = LinearAttentionRecurrent(dim, n_heads)
standard_attn = StandardAttention(dim, n_heads)

x = torch.randn(1, 64, dim)
with torch.no_grad():
    out_linear = linear_attn(x)
    out_rec = linear_attn_rec(x)
    out_std = standard_attn(x)

print(f"=== 线性注意力 vs 标准注意力 ===")
print(f"输入: {x.shape}")
print(f"标准注意力输出: {out_std.shape}")
print(f"线性注意力(并行)输出: {out_linear.shape}")
print(f"线性注意力(递推)输出: {out_rec.shape}")

diff = (out_linear - out_rec).abs().max().item()
print(f"\n并行vs递推实现最大差异: {diff:.6f} (应接近0)")

print(f"\n--- 推理速度对比 ---")
for seq_len in [64, 128, 256, 512, 1024]:
    x = torch.randn(1, seq_len, dim)
    with torch.no_grad():
        start = time.perf_counter()
        for _ in range(20):
            _ = standard_attn(x)
        std_time = (time.perf_counter() - start) / 20 * 1000

        start = time.perf_counter()
        for _ in range(20):
            _ = linear_attn(x)
        lin_time = (time.perf_counter() - start) / 20 * 1000

    print(f"  seq={seq_len:<6} 标准={std_time:.3f}ms 线性={lin_time:.3f}ms 比值={std_time/lin_time:.2f}x")

print(f"\n线性注意力优势: 复杂度O(Nd²) vs 标准O(N²d)")
print(f"当N >> d时，线性注意力显著更快")

---
### 状态空间模型（State Space Models, SSM）

#### 什么是SSM？

通过隐状态的线性递推建模序列依赖，推理时每步仅需$O(d)$计算和$O(d)$内存，不受序列长度影响。Mamba引入选择性机制，使SSM具备输入依赖的建模能力。

#### 为什么SSM有效？

1. **线性递推**：每步仅需$O(d)$计算，远低于注意力的$O(nd)$
2. **并行训练**：虽然推理是递推的，但训练时可通过卷积模式并行
3. **长程记忆**：通过选择机制（Mamba）可以高效捕获长程依赖

#### SSM的数学原理

连续时间SSM：
$$\dot{h}(t) = Ah(t) + Bx(t), \quad y(t) = Ch(t) + Dx(t)$$

离散化（零阶保持）：
$$h_t = \bar{A}h_{t-1} + \bar{B}x_t, \quad y_t = Ch_t + Dx_t$$

其中：
- $h_t \in \mathbb{R}^{d}$：隐状态
- $\bar{A} = e^{\Delta A}$：离散化状态转移矩阵
- $\bar{B} = (\Delta A)^{-1}(e^{\Delta A} - I) \cdot \Delta B$：离散化输入矩阵
- $\Delta$：步长（可学习）

Mamba的选择机制：$B, C, \Delta$依赖于输入$x_t$，使模型能选择性地传播或遗忘信息

In [ ]:
class S6Layer(nn.Module):
    def __init__(self, dim, state_dim=16, dt_min=0.001, dt_max=0.1):
        super().__init__()
        self.dim = dim
        self.state_dim = state_dim

        A = torch.arange(1, state_dim + 1, dtype=torch.float32).unsqueeze(0).expand(dim, -1).clone()
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(dim))

        self.x_proj = nn.Linear(dim, state_dim * 2, bias=False)
        self.dt_proj = nn.Linear(state_dim, dim, bias=True)

        init_dt = torch.exp(
            torch.rand(dim, state_dim) * (np.log(dt_max) - np.log(dt_min)) + np.log(dt_min)
        )
        with torch.no_grad():
            self.dt_proj.weight.copy_(init_dt)

    def forward(self, x):
        B_batch, N, D = x.shape
        A = -torch.exp(self.A_log)

        x_dbl = self.x_proj(x)
        B_proj = x_dbl[:, :, :self.state_dim]
        C_proj = x_dbl[:, :, self.state_dim:]

        dt = F.softplus(self.dt_proj(B_proj))

        dA = torch.exp(dt * A)
        dB = dt.unsqueeze(-1) * B_proj.unsqueeze(2)

        h = torch.zeros(B_batch, D, self.state_dim, device=x.device)
        outputs = []

        for t in range(N):
            h = h * dA[:, t, :, :] + x[:, t, :].unsqueeze(-1) * dB[:, t, :, :]
            y_t = (h * C_proj[:, t, :].unsqueeze(1)).sum(dim=-1) + x[:, t, :] * self.D
            outputs.append(y_t)

        return torch.stack(outputs, dim=1)

class SelectiveSSMBlock(nn.Module):
    def __init__(self, dim, state_dim=16, expand_factor=2):
        super().__init__()
        self.dim = dim
        self.inner_dim = dim * expand_factor
        self.in_proj = nn.Linear(dim, self.inner_dim * 2, bias=False)
        self.conv1d = nn.Conv1d(
            self.inner_dim, self.inner_dim,
            kernel_size=3, padding=1, groups=self.inner_dim
        )
        self.ssm = S6Layer(self.inner_dim, state_dim)
        self.out_proj = nn.Linear(self.inner_dim, dim, bias=False)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        residual = x
        x = self.norm(x)
        projected = self.in_proj(x)
        x1, x2 = projected.chunk(2, dim=-1)
        x1 = self.conv1d(x1.transpose(1, 2)).transpose(1, 2)
        x1 = F.silu(x1)
        x1 = self.ssm(x1)
        out = x1 * F.silu(x2)
        return self.out_proj(out) + residual

dim = 64
ssm_block = SelectiveSSMBlock(dim, state_dim=16, expand_factor=2)
x = torch.randn(2, 32, dim)
with torch.no_grad():
    out = ssm_block(x)

print(f"=== SSM (Mamba风格) ===")
print(f"输入: {x.shape}")
print(f"输出: {out.shape}")
print(f"参数量: {sum(p.numel() for p in ssm_block.parameters()):,}")

print(f"\n--- SSM vs Attention 推理复杂度对比 ---")
print(f"{'序列长度':<12} {'Attention O(N²)':<20} {'SSM O(N)':<20}")
print("-" * 52)
for N in [128, 256, 512, 1024, 2048, 4096, 8192]:
    attn_ops = N * N
    ssm_ops = N * 16 * 64
    print(f"{N:<12} {attn_ops:<20,} {ssm_ops:<20,}")

print(f"\nSSM核心优势:")
print(f"1. 推理时O(1)每步（固定状态大小），不受序列长度影响")
print(f"2. 无需KV Cache，内存占用恒定")
print(f"3. 选择性机制使SSM具备输入依赖的建模能力")

---
### SSM训练：卷积模式并行

#### 为什么需要卷积模式？

递推模式无法并行训练（每步依赖上一步），但SSM的线性结构允许将递推展开为卷积：
$$y = K * x$$

其中SSM卷积核$K = (C\bar{B}, C\bar{A}\bar{B}, C\bar{A}^2\bar{B}, \ldots)$

#### 两种模式对比

| 模式 | 训练 | 推理 | 复杂度 |
|------|------|------|--------|
| 递推 | 串行 | 流式 | $O(Nd^2)$ |
| 卷积 | 并行 | 非流式 | $O(Nd^2\log N)$ |

训练用卷积模式（并行），推理用递推模式（流式），两者数学等价。

In [ ]:
class SSMConvMode(nn.Module):
    def __init__(self, dim, state_dim=16):
        super().__init__()
        self.dim = dim
        self.state_dim = state_dim
        A = torch.arange(1, state_dim + 1, dtype=torch.float32).unsqueeze(0).expand(dim, -1).clone()
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(dim))
        self.B_proj = nn.Linear(dim, state_dim, bias=False)
        self.C_proj = nn.Linear(dim, state_dim, bias=False)
        self.dt_proj = nn.Linear(dim, dim, bias=True)

    def _compute_kernel(self, L):
        A = -torch.exp(self.A_log)
        return A, L

    def forward_conv(self, x):
        B_batch, N, D = x.shape
        A = -torch.exp(self.A_log)
        B = self.B_proj(x)
        C = self.C_proj(x)
        dt = F.softplus(self.dt_proj(x))

        dA = torch.exp(dt * A)
        dB = dt.unsqueeze(-1) * B.unsqueeze(2)

        h = torch.zeros(B_batch, D, self.state_dim, device=x.device)
        outputs = []
        for t in range(N):
            h = h * dA[:, t, :, :] + x[:, t, :].unsqueeze(-1) * dB[:, t, :, :]
            y_t = (h * C[:, t, :].unsqueeze(1)).sum(dim=-1) + x[:, t, :] * self.D
            outputs.append(y_t)
        return torch.stack(outputs, dim=1)

    def forward_recurrent(self, x, state=None):
        B_batch, D = x.shape
        A = -torch.exp(self.A_log)
        B = self.B_proj(x)
        C = self.C_proj(x)
        dt = F.softplus(self.dt_proj(x))

        dA = torch.exp(dt * A)
        dB = dt.unsqueeze(-1) * B.unsqueeze(2)

        if state is None:
            state = torch.zeros(B_batch, D, self.state_dim, device=x.device)

        state = state * dA[:, 0, :, :] + x.unsqueeze(-1) * dB[:, 0, :, :]
        y = (state * C[:, 0, :].unsqueeze(1)).sum(dim=-1) + x * self.D
        return y, state

ssm_conv = SSMConvMode(dim=64, state_dim=16)
x = torch.randn(2, 32, 64)

with torch.no_grad():
    out_conv = ssm_conv.forward_conv(x)

    state = None
    recurrent_outputs = []
    for t in range(32):
        y_t, state = ssm_conv.forward_recurrent(x[:, t, :], state)
        recurrent_outputs.append(y_t)
    out_rec = torch.stack(recurrent_outputs, dim=1)

max_diff = (out_conv - out_rec).abs().max().item()
print(f"=== SSM 卷积模式 vs 递推模式 ===")
print(f"卷积模式输出: {out_conv.shape}")
print(f"递推模式输出: {out_rec.shape}")
print(f"两种模式最大差异: {max_diff:.6f} (应接近0)")

print(f"\n--- 递推模式流式推理演示 ---")
ssm_conv.eval()
state = None
with torch.no_grad():
    for step in range(5):
        x_step = torch.randn(1, 64)
        y_step, state = ssm_conv.forward_recurrent(x_step, state)
        print(f"  Step {step+1}: 输入{x_step.shape} -> 输出{y_step.shape}, 状态{state.shape}")

print(f"\n流式推理关键: 状态大小固定为(D, state_dim)，不随序列增长")
print(f"适合端侧部署: 恒定内存，恒定延迟")

---
### 混合架构（Hybrid Architecture）

#### 什么是混合架构？

将注意力层和SSM/线性注意力层交替堆叠，在关键位置保留精确注意力，其余位置使用高效线性层。如Jamba、Zamba等模型。

#### 为什么混合架构有效？

1. **精确+高效**：注意力层提供精确的长程依赖建模，SSM层提供高效的序列处理
2. **精度接近纯注意力**：少量注意力层（如每4层1个）即可保持接近纯注意力模型的精度
3. **推理效率高**：大部分层是SSM，推理速度和内存占用接近纯SSM模型

#### 混合架构的数学原理

混合层的计算：
$$y = \begin{cases} \text{SSM}(x) & \text{if layer } l \mod s \neq 0 \\ \text{Attention}(x) + \text{SSM}(x) & \text{if layer } l \mod s = 0 \end{cases}$$

其中$s$为注意力间隔（如$s=4$表示每4层一个注意力层）。

计算复杂度：$O(\frac{L}{s} n^2 d + L \cdot n d^2)$，当$s$较大时接近$O(Lnd^2)$

In [ ]:
class HybridBlock(nn.Module):
    def __init__(self, dim, block_type='ssm', n_heads=4, state_dim=16):
        super().__init__()
        self.block_type = block_type
        if block_type == 'attention':
            self.attn = nn.MultiheadAttention(dim, n_heads, batch_first=True)
            self.ffn = nn.Sequential(nn.Linear(dim, dim*4), nn.SiLU(), nn.Linear(dim*4, dim))
            self.ln1 = nn.LayerNorm(dim)
            self.ln2 = nn.LayerNorm(dim)
        else:
            self.ssm = SelectiveSSMBlock(dim, state_dim=state_dim)

    def forward(self, x):
        if self.block_type == 'attention':
            h = self.ln1(x)
            h, _ = self.attn(h, h, h)
            x = x + h
            x = x + self.ffn(self.ln2(x))
            return x
        else:
            return self.ssm(x)

class HybridModel(nn.Module):
    def __init__(self, dim=64, n_layers=8, attn_interval=4, n_heads=4, state_dim=16):
        super().__init__()
        self.blocks = nn.ModuleList()
        for i in range(n_layers):
            if (i + 1) % attn_interval == 0:
                self.blocks.append(HybridBlock(dim, 'attention', n_heads, state_dim))
            else:
                self.blocks.append(HybridBlock(dim, 'ssm', n_heads, state_dim))

    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        return x

dim = 64
hybrid = HybridModel(dim=dim, n_layers=8, attn_interval=4, n_heads=4, state_dim=16)

n_attn = sum(1 for b in hybrid.blocks if b.block_type == 'attention')
n_ssm = sum(1 for b in hybrid.blocks if b.block_type == 'ssm')

print(f"=== 混合架构 (Jamba风格) ===")
print(f"总层数: 8")
print(f"Attention层: {n_attn} (第4,8层)")
print(f"SSM层: {n_ssm} (其余层)")
print(f"\n架构可视化:")
for i, block in enumerate(hybrid.blocks):
    tag = 'Attention' if block.block_type == 'attention' else 'SSM'
    print(f"  Layer {i+1}: {tag}")

x = torch.randn(2, 32, dim)
with torch.no_grad():
    out = hybrid(x)
print(f"\n输入: {x.shape}, 输出: {out.shape}")

print(f"\n--- 不同序列长度的推理时间对比 ---")
attn_only = nn.ModuleList([HybridBlock(dim, 'attention', 4) for _ in range(8)])
ssm_only = nn.ModuleList([HybridBlock(dim, 'ssm', 4, 16) for _ in range(8)])

for seq_len in [64, 128, 256, 512]:
    x = torch.randn(1, seq_len, dim)
    with torch.no_grad():
        start = time.perf_counter()
        for block in attn_only:
            x_a = block(x)
        attn_t = (time.perf_counter() - start) * 1000

        start = time.perf_counter()
        for block in ssm_only:
            x_s = block(x)
        ssm_t = (time.perf_counter() - start) * 1000

        start = time.perf_counter()
        for block in hybrid.blocks:
            x_h = block(x)
        hybrid_t = (time.perf_counter() - start) * 1000

    print(f"  N={seq_len:<5} Attention={attn_t:.2f}ms SSM={ssm_t:.2f}ms Hybrid={hybrid_t:.2f}ms")

print(f"\n混合架构优势: 在关键位置保留精确注意力，其余位置用SSM降低复杂度")

---
### 线性RNN架构：RWKV风格

#### 什么是RWKV？

RWKV使用线性递推替代传统RNN的非线性激活，结合时间混合（Time Mixing）和通道混合（Channel Mixing）交替，实现$O(1)$推理复杂度和可并行训练。

#### 为什么RWKV有效？

1. **O(1)推理**：每步仅需维护固定大小的隐状态，不随序列增长
2. **可并行训练**：训练时将递推展开为并行扫描（parallel scan）
3. **注意力兼容**：时间混合模块使用类似注意力的加权求和，但权重是递推计算的

#### RWKV的数学原理

**时间混合（WKV注意力）**：
$$wkv_t = \frac{\sum_{i=1}^{t-1} e^{-(t-1-i)w + u_i} \cdot v_i}{\sum_{i=1}^{t-1} e^{-(t-1-i)w + u_i}}$$

递推形式：
$$wkv_t = \frac{e^{u_{t-1}} \cdot v_{t-1} + e^{-w} \cdot a_{t-1}}{e^{u_{t-1}} + e^{-w} \cdot b_{t-1}}$$

其中：
- $w$：时间衰减因子（可学习），控制历史信息的重要性
- $u_i$：位置奖励（可学习），给当前位置额外权重
- $a_t, b_t$：递推累积量，$a_t = e^{u_{t-1}} v_{t-1} + e^{-w} a_{t-1}$
- 输出：$o_t = \sigma(r_t) \cdot (wkv_t \cdot W_o + x_t \cdot W_{o,r})$

In [ ]:
class RWKVTimeMixing(nn.Module):
    def __init__(self, dim, n_heads=4):
        super().__init__()
        self.dim = dim
        self.n_heads = n_heads
        self.head_dim = dim // n_heads

        self.time_decay = nn.Parameter(torch.randn(n_heads) * 0.1 - 5.0)
        self.time_first = nn.Parameter(torch.randn(n_heads) * 0.1)

        self.key = nn.Linear(dim, dim, bias=False)
        self.value = nn.Linear(dim, dim, bias=False)
        self.receptance = nn.Linear(dim, dim, bias=False)
        self.output = nn.Linear(dim, dim, bias=False)

    def forward(self, x):
        B, N, C = x.shape
        r = torch.sigmoid(self.receptance(x)).view(B, N, self.n_heads, self.head_dim)
        k = self.key(x).view(B, N, self.n_heads, self.head_dim)
        v = self.value(x).view(B, N, self.n_heads, self.head_dim)

        w = torch.exp(-torch.exp(self.time_decay)).view(1, 1, self.n_heads, 1)
        u = self.time_first.view(1, 1, self.n_heads, 1)

        outputs = []
        aa = torch.zeros(B, self.n_heads, self.head_dim, device=x.device)
        bb = torch.zeros(B, self.n_heads, self.head_dim, device=x.device)
        pp = torch.full((B, self.n_heads, self.head_dim), -1e30, device=x.device)

        for t in range(N):
            kk = k[:, t, :, :]
            vv = v[:, t, :, :]
            rr = r[:, t, :, :]

            ww = pp + u
            e1 = torch.exp(ww - torch.max(ww, pp))
            e2 = torch.exp(pp - torch.max(ww, pp))
            wkv = (e1 * vv + e2 * aa) / (e1 + e2 * bb + 1e-8)

            pp_new = pp * w + kk
            aa_new = e1 * vv + e2 * aa * w
            bb_new = e1 + e2 * bb * w

            out_t = rr * wkv
            outputs.append(out_t)

            pp, aa, bb = pp_new, aa_new, bb_new

        out = torch.stack(outputs, dim=1).reshape(B, N, C)
        return self.output(out)

class RWKVChannelMixing(nn.Module):
    def __init__(self, dim, hidden_factor=4):
        super().__init__()
        hidden = dim * hidden_factor
        self.key = nn.Linear(dim, hidden, bias=False)
        self.value = nn.Linear(hidden, dim, bias=False)
        self.receptance = nn.Linear(dim, dim, bias=False)

    def forward(self, x):
        r = torch.sigmoid(self.receptance(x))
        k = torch.relu(self.key(x)) ** 2
        return r * self.value(k)

dim = 128
rwkv_tm = RWKVTimeMixing(dim, n_heads=4)
rwkv_cm = RWKVChannelMixing(dim)

x = torch.randn(2, 64, dim)
with torch.no_grad():
    tm_out = rwkv_tm(x)
    cm_out = rwkv_cm(x)

print(f"=== RWKV线性RNN架构 ===")
print(f"输入: {x.shape}")
print(f"时间混合输出: {tm_out.shape}")
print(f"通道混合输出: {cm_out.shape}")

tm_params = sum(p.numel() for p in rwkv_tm.parameters())
cm_params = sum(p.numel() for p in rwkv_cm.parameters())
print(f"\n参数量:")
print(f"  时间混合: {tm_params:,}")
print(f"  通道混合: {cm_params:,}")

print(f"\nRWKV vs Transformer vs SSM:")
comparisons = [
    ('Transformer', 'O(n²)', 'O(n)', '并行训练, 精确注意力'),
    ('SSM (Mamba)', 'O(n)', 'O(1)', '线性训练, O(1)推理'),
    ('RWKV', 'O(n)', 'O(1)', '线性训练, O(1)推理, 类注意力'),
]
print(f"  {'架构':<15} {'训练复杂度':<12} {'推理复杂度':<12} {'特点':<25}")
print("  " + "-" * 64)
for name, train, infer, feat in comparisons:
    print(f"  {name:<15} {train:<12} {infer:<12} {feat:<25}")

---
### 线性注意力端侧部署实战

#### 端侧部署的关键考量

1. **内存预算**：端侧设备内存有限，KV Cache是主要瓶颈
2. **延迟要求**：实时应用需要每步推理在毫秒级完成
3. **算子支持**：SSM的递推算子可能不被所有推理框架支持

#### 各架构的端侧适配策略

| 架构 | 内存占用 | 延迟 | 部署难度 | 适用场景 |
|------|---------|------|---------|----------|
| 标准Transformer | O(n) KV Cache | 高 | 低 | 短序列 |
| 线性注意力 | O(d²) 固定 | 中 | 中 | 中长序列 |
| SSM (Mamba) | O(d) 固定 | 低 | 高 | 长序列流式 |
| RWKV | O(d) 固定 | 低 | 高 | 长序列流式 |
| 混合架构 | 介于中间 | 中 | 高 | 通用 |

In [ ]:
def benchmark_kv_cache(dim=512, n_heads=8, n_kv_heads=2, seq_len=1024, batch_size=1, dtype_bytes=2):
    head_dim = dim // n_heads
    n_rep = n_heads // n_kv_heads

    full_kv = batch_size * seq_len * 2 * n_heads * head_dim * dtype_bytes
    gqa_kv = batch_size * seq_len * 2 * n_kv_heads * head_dim * dtype_bytes
    linear_attn_state = batch_size * n_heads * head_dim * head_dim * dtype_bytes
    ssm_state = batch_size * dim * 16 * dtype_bytes

    return {
        'full_attn_kv': full_kv,
        'gqa_kv': gqa_kv,
        'linear_attn': linear_attn_state,
        'ssm': ssm_state,
    }

print(f"=== 端侧内存占用对比 (dim=512, seq=1024, FP16) ===")
print(f"{'架构':<25} {'内存(KB)':<15} {'相对标准注意力':<15}")
print("-" * 55)

mem = benchmark_kv_cache()
baseline = mem['full_attn_kv']
for name, size in mem.items():
    label = {
        'full_attn_kv': '标准注意力KV Cache',
        'gqa_kv': 'GQA KV Cache',
        'linear_attn': '线性注意力状态',
        'ssm': 'SSM隐状态',
    }[name]
    print(f"{label:<25} {size/1024:<15.1f} {size/baseline:<15.3f}")

print(f"\n--- 不同序列长度的KV Cache增长 ---")
print(f"{'序列长度':<12} {'标准注意力':<15} {'GQA':<15} {'线性注意力':<15} {'SSM':<15}")
print("-" * 72)
for seq_len in [128, 256, 512, 1024, 2048, 4096]:
    mem = benchmark_kv_cache(seq_len=seq_len)
    print(f"{seq_len:<12} {mem['full_attn_kv']/1024:<15.1f} {mem['gqa_kv']/1024:<15.1f} {mem['linear_attn']/1024:<15.1f} {mem['ssm']/1024:<15.1f}")

print(f"\n关键结论:")
print(f"1. 标准注意力KV Cache随序列线性增长，长序列是内存瓶颈")
print(f"2. GQA减少KV头数，KV Cache降至{2/8*100:.0f}%")
print(f"3. 线性注意力/SSM状态大小与序列长度无关，适合长序列")
print(f"4. SSM状态最小，但需要自定义算子支持")

print(f"\n=== 端侧部署建议 ===")
print(f"1. 短序列(<512): 标准Transformer + GQA + 量化，生态最成熟")
print(f"2. 中长序列(512-4K): 线性注意力或混合架构，平衡精度和效率")
print(f"3. 超长序列(>4K): SSM/Mamba，恒定内存和延迟")
print(f"4. 流式推理: SSM/RWKV递推模式，每步O(1)内存")
print(f"5. 框架选择: ONNX导出 + 自定义算子注册(递推SSM/RWKV)")

---
### 线性注意力与SSM总结

| 架构 | 训练复杂度 | 推理复杂度 | 内存 | 精度 | 部署难度 |
|------|-----------|-----------|------|------|----------|
| 标准Transformer | O(n²d) | O(nd) per step | O(n) KV | 最高 | 低 |
| 线性注意力 | O(nd²) | O(d²) per step | O(d²) 固定 | 中高 | 中 |
| SSM (Mamba) | O(nd²) | O(d) per step | O(d) 固定 | 中 | 高 |
| RWKV | O(nd²) | O(d) per step | O(d) 固定 | 中 | 高 |
| 混合架构 | 介于中间 | 介于中间 | 介于中间 | 高 | 高 |

**端侧部署最佳实践**：
1. 根据序列长度选择架构：短序列用标准注意力，长序列用SSM
2. 混合架构是通用方案：关键层用注意力，其余用SSM
3. 递推模式是端侧推理的关键：恒定内存和延迟
4. 需要自定义算子支持SSM/RWKV的递推计算
5. 量化仍适用：SSM的线性层和投影层可量化为INT8